# AG04 · Planning & Reasoning Patterns

**Track:** Agents (lesson 4 of 15) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free, or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** AG02 (the loop), AG03 (tools) · **You will build:** a constraint-checked trip planner solved three ways (ReAct, plan-and-execute, reflect-with-verifier) and a scored comparison

---
## 🎯 Why this matters
The loop from AG02 lets a model *act*; it does not make it *think well before and after acting*. Long tasks fail in recognisable ways: the model forgets a constraint it read ten turns ago, commits to a plan built on a stale fact, or declares victory without checking. The **reasoning pattern** you wrap around the loop (when to think, when to plan, when to verify) is the main lever you have besides the tools themselves.

**Real scenario.** *Southern Lakes Events* plans 3-day offsites in Queenstown. A coordinator gives the agent a brief: budget per person, a rain forecast, opening hours, "at least one cultural activity and two adventure ones", and no more than 9 hours a day including driving. A human planner checks all that in their head and still gets it wrong 1 time in 5. You will build a *machine-checkable* version of the brief and compare three strategies on one scored task.

## 🧭 Learning objectives
By the end you can:
1. Explain ReAct, plan-and-execute and reflection, and say which failure each one targets.
2. Encode a brief as an executable **verifier** and use it as a tool, a scorer and a critique signal.
3. Implement a plan validator, a plan executor with step dependencies, and a bounded reflection loop with cycle detection.
4. Compare strategies on **score, model calls, tool calls**, and explain why offline numbers here illustrate the harness rather than benchmark any model.
5. Decide when a thinking model makes a scaffold unnecessary, using an ablation instead of intuition.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: three ways to spend extra computation
All three patterns spend more model calls (or tokens) to buy reliability. They differ in *where* the extra thinking goes:

```
 ReAct                 interleave:   think → act → observe → think → act → …      (decide as you learn)
 Plan-and-execute      separate:     PLAN once ──▶ run step 1 … step n ──▶ verify  (decide, then commit)
 Reflection            iterate:      draft ──▶ critique (with evidence) ──▶ revise ──▶ …
```

| Pattern | Idea | Strong when | Fails when |
|---|---|---|---|
| **ReAct** (Yao et al., 2022) | Alternate reasoning text with tool calls; each observation shapes the next thought. This *is* the AG02 loop. | Path depends on what tools return; few steps | Long runs drift; no global view, easy to forget constraints |
| **Plan-and-execute** | A planner writes the whole step list first; an executor runs it (steps can be cheap/parallel/deterministic). | Steps are knowable up front; you want an auditable, approvable plan; cheaper executors | The world changes mid-run and the plan goes **stale**; planner mistakes are executed faithfully |
| **Reflection / self-critique** (Shinn et al. *Reflexion*, Madaan et al. *Self-Refine*, 2023) | Produce, critique, revise. | There is a *checkable* quality bar | The critique has **no external signal**, in which case models often cannot find their own errors (Huang et al., 2023) |

**The unifying principle: verification.** In every pattern, the biggest gain comes from a step that *checks the work against ground truth*, not from the model re-reading its own prose. A verifier can be code (our itinerary checker), a test-suite, a database lookup or a human. If you cannot write one, ask whether the task is ready for an agent (AG01's viability gate).

**Thinking models.** Models with built-in extended reasoning do much of the "think before you answer" internally. That *replaces* hand-rolled chain-of-thought prompting and often simple planning scaffolds; it does **not** replace tools, external verification, approvals, or budget control. We return to this in §6.

## 2 · The task, as code: a trip brief with a verifier
The brief (synthetic data; names are real places but prices, hours and opening days are invented for the exercise):

* 3 days, base hotel in **Queenstown (town)**; travel by road to **Arrowtown** and **Glenorchy** and back to the hotel each day.
* **C1** total activity cost ≤ NZD 600 per person · **C2** each day ≤ 9 h including travel · **C3** no outdoor activity on rainy days (forecast: day 2) · **C4** no repeated activity
* **C5** ≥ 1 cultural activity · **C6** ≥ 2 adventure activities · **C7** activities must exist and be open that day · **C8** 2-3 activities per day

The checker returns *specific, located* violations (`[C2] day 3: 11.2 h …`). Specificity is what makes it usable as a critique signal.

In [ ]:
import json, re, itertools, collections
from dataclasses import dataclass
from typing import Literal
from agentkit import tool, ToolBox, get_llm, say, call, calls, run_agent, is_live

ACT = {  # id: (zone, hours, cost_pp NZD, kind, category, open_days)
    "shotover_jet":        ("town",      1.0, 145, "outdoor", "adventure", (1, 2, 3)),
    "skyline_luge":        ("town",      2.0,  70, "outdoor", "adventure", (1, 2, 3)),
    "kawarau_bungy":       ("arrowtown", 1.5, 205, "outdoor", "adventure", (1, 2, 3)),
    "routeburn_hike":      ("glenorchy", 7.0,   0, "outdoor", "adventure", (1, 3)),
    "glenorchy_boardwalk": ("glenorchy", 1.5,   0, "outdoor", "nature",    (1, 2, 3)),
    "kiwi_birdlife_park":  ("town",      1.5,  50, "outdoor", "nature",    (1, 2, 3)),
    "wine_tour":           ("arrowtown", 4.0, 120, "indoor",  "food",      (2, 3)),
    "lakes_museum":        ("arrowtown", 1.5,  15, "indoor",  "culture",   (1, 2)),
    "pottery_workshop":    ("town",      2.0,  90, "indoor",  "culture",   (1, 2, 3)),
    "spa_afternoon":       ("town",      3.0, 150, "indoor",  "wellness",  (2, 3)),
    "burger_lunch":        ("town",      1.0,  25, "indoor",  "food",      (1, 2, 3)),
    "cooking_class":       ("town",      2.5, 110, "indoor",  "food",      (1, 2, 3)),
}
FIELDS = ("zone", "hours", "cost_pp", "kind", "category", "open_days")
ACTS = {k: dict(zip(FIELDS, v)) for k, v in ACT.items()}
TRAVEL_MIN = {frozenset(("town", "arrowtown")): 25, frozenset(("town", "glenorchy")): 45, frozenset(("arrowtown", "glenorchy")): 70}
WEATHER = {1: "clear", 2: "rain", 3: "clear"}
BUDGET, MAX_DAY_H, DAYS = 600, 9.0, (1, 2, 3)

def travel(a, b): return 0 if a == b else TRAVEL_MIN[frozenset((a, b))]

@dataclass
class Violation:
    code: str; day: int | None; msg: str
    def __str__(self): return f"[{self.code}]" + (f" day {self.day}:" if self.day else "") + f" {self.msg}"

def day_hours(acts):
    known = [a for a in acts if a in ACTS]
    zones = ["town"] + [ACTS[a]["zone"] for a in known] + ["town"]
    return sum(ACTS[a]["hours"] for a in known) + sum(travel(x, y) for x, y in zip(zones, zones[1:])) / 60

def validate(itin: dict) -> list:
    """itin: {day:int -> [activity ids]}. Returns a list of located Violations (empty = brief satisfied)."""
    v, allacts = [], [a for d in DAYS for a in itin.get(d, [])]
    cost = sum(ACTS[a]["cost_pp"] for a in allacts if a in ACTS)
    if cost > BUDGET: v.append(Violation("C1", None, f"total cost ${cost} exceeds budget ${BUDGET}"))
    for d in DAYS:
        acts = itin.get(d, [])
        if not 2 <= len(acts) <= 3: v.append(Violation("C8", d, f"{len(acts)} activities; need 2-3"))
        h = day_hours(acts)
        if h > MAX_DAY_H: v.append(Violation("C2", d, f"{h:.1f} h including travel exceeds {MAX_DAY_H:.0f} h"))
        for a in acts:
            if a not in ACTS: v.append(Violation("C7", d, f"unknown activity '{a}'")); continue
            if d not in ACTS[a]["open_days"]: v.append(Violation("C7", d, f"'{a}' is closed on day {d}"))
            if WEATHER[d] == "rain" and ACTS[a]["kind"] == "outdoor": v.append(Violation("C3", d, f"outdoor '{a}' but rain is forecast"))
    for a, n in collections.Counter(allacts).items():
        if n > 1: v.append(Violation("C4", None, f"'{a}' appears {n} times"))
    known = [a for a in allacts if a in ACTS]
    if not any(ACTS[a]["category"] == "culture" for a in known): v.append(Violation("C5", None, "no cultural activity"))
    if sum(ACTS[a]["category"] == "adventure" for a in set(known)) < 2: v.append(Violation("C6", None, "fewer than 2 adventure activities"))
    return v

def score(itin) -> float:
    """Fraction of the 8 constraints with no violation (0 if the answer is unusable)."""
    if itin is None: return 0.0
    return (8 - len({x.code for x in validate(itin)})) / 8

def parse_itinerary(text: str) -> dict:
    m = re.search(r"\{.*\}", text, re.S)
    if not m: raise ValueError("no JSON object found")
    raw = json.loads(m.group(0))
    return {int(k): list(v) for k, v in raw.items()}

# --- the verifier has to be trustworthy: prove the task is feasible and the checker is not vacuous ---
def solve():
    def day_ok(d, acts):
        return (2 <= len(acts) <= 3 and day_hours(acts) <= MAX_DAY_H and
                all(d in ACTS[a]["open_days"] and not (WEATHER[d] == "rain" and ACTS[a]["kind"] == "outdoor") for a in acts))
    zone_order = {"town": 0, "arrowtown": 1, "glenorchy": 2}
    opts = {d: [c for n in (2, 3) for c in itertools.combinations(sorted(ACTS, key=lambda a: zone_order[ACTS[a]["zone"]]), n) if day_ok(d, c)] for d in DAYS}
    for a, b, c in itertools.product(opts[1], opts[2], opts[3]):
        itin = {1: list(a), 2: list(b), 3: list(c)}
        if not validate(itin): return itin
GOLD = solve()
assert GOLD is not None and validate(GOLD) == [] and score(GOLD) == 1.0
print("a valid itinerary exists, e.g.:", GOLD)
bad = {1: ["shotover_jet", "skyline_luge"], 2: ["kawarau_bungy", "lakes_museum"], 3: ["routeburn_hike", "spa_afternoon"]}
print(*validate(bad), sep="\n")
assert {x.code for x in validate(bad)} >= {"C2", "C3"}

## 3 · The three strategies on one scored task
We give every strategy the same words as the brief and the same scorer. A tiny harness counts **model calls** and **tool calls** so we can talk about cost, not just quality.

**Offline honesty note.** In OFFLINE mode the model's outputs are *scripted* (plausible traces of what an LLM does), except the reviser in §3.3 which is a small local-search program. So the *scores* below demonstrate the mechanics and the scoring pipeline; they are **not** evidence about any model. Run LIVE and the identical code produces real numbers; compare then.

### Shared pieces: tools, prompt, counting wrapper

In [ ]:
@tool
def list_activities(zone: Literal["any", "town", "arrowtown", "glenorchy"] = "any") -> list:
    """List bookable activities with zone, hours, cost per person (NZD), indoor/outdoor, category and open days. Use before choosing activities. Returns a JSON list; filter by zone to shorten it.

    Args:
        zone: Only activities in this zone, or any for all.
    """
    return [{"id": k, **v} for k, v in ACTS.items() if zone in ("any", v["zone"])]

@tool
def get_weather(day: int) -> str:
    """Get the forecast for one trip day. Use before scheduling outdoor activities. Returns "clear" or "rain".

    Args:
        day: Trip day number, 1 to 3.
    """
    if day not in WEATHER: raise ValueError(f"day {day} is outside the trip; use 1, 2 or 3.")
    return WEATHER[day]

@tool
def travel_minutes(from_zone: Literal["town", "arrowtown", "glenorchy"], to_zone: Literal["town", "arrowtown", "glenorchy"]) -> int:
    """Driving minutes between two zones. Use to check a day's total time. Returns minutes as an integer.

    Args:
        from_zone: Starting zone.
        to_zone: Destination zone.
    """
    return travel(from_zone, to_zone)

@tool
def check_itinerary(itinerary_json: str) -> str:
    """Verify an itinerary against the brief. Use after drafting or revising. Returns OK, or one located violation per line.

    Args:
        itinerary_json: JSON object mapping day number to a list of activity ids, e.g. {"1": ["shotover_jet", "burger_lunch"], "2": [...], "3": [...]}.
    """
    try: itin = parse_itinerary(itinerary_json)
    except (ValueError, json.JSONDecodeError) as e: raise ValueError(f"could not read itinerary: {e}. Send a JSON object like {{\"1\": [...], \"2\": [...], \"3\": [...]}}.")
    v = validate(itin)
    return "OK" if not v else "\n".join(str(x) for x in v)

DATA_TOOLS = ToolBox(list_activities, get_weather, travel_minutes)
ALL_TOOLS = ToolBox(list_activities, get_weather, travel_minutes, check_itinerary)

BRIEF = ("Plan a 3-day Queenstown offsite for our team. Hotel is in town; drive to Arrowtown/Glenorchy and back each day. "
         f"Constraints: total activity cost <= ${BUDGET} per person; each day <= {MAX_DAY_H:.0f} hours including driving; no outdoor activity on rainy days; "
         "no repeated activity; at least 1 cultural and 2 adventure activities in total; each activity must be open that day; 2-3 activities per day. "
         'Answer with ONLY a JSON object {"1": [ids], "2": [ids], "3": [ids]}.')

class Counted:
    """Wraps any agentkit LLM and counts model calls."""
    def __init__(self, llm): self.llm, self.n, self.model, self.mode = llm, 0, getattr(llm, "model", ""), getattr(llm, "mode", "")
    def chat(self, *a, **k): self.n += 1; return self.llm.chat(*a, **k)

def J(itin): return json.dumps({str(d): v for d, v in itin.items()})
RESULTS = {}   # strategy -> dict(score, llm_calls, tool_calls, violations)

### 3.1 · Baseline: single-shot
One call, the whole catalog and forecast in the prompt, no tools. Cheapest and the right baseline: *always measure a scaffold against it.*

In [ ]:
catalog_text = "\n".join(f"{k}: zone={v['zone']}, {v['hours']}h, ${v['cost_pp']}, {v['kind']}, {v['category']}, open days {list(v['open_days'])}" for k, v in ACTS.items())
single_prompt = f"{BRIEF}\n\nForecast: {WEATHER}\nDriving minutes: town-arrowtown 25, town-glenorchy 45, arrowtown-glenorchy 70.\nActivities:\n{catalog_text}"
DRAFT = {1: ["shotover_jet", "skyline_luge", "burger_lunch"], 2: ["kawarau_bungy", "wine_tour", "lakes_museum"],
         3: ["routeburn_hike", "glenorchy_boardwalk", "spa_afternoon"]}          # recorded trace of a plausible single-shot answer

llm1 = Counted(get_llm(script=[say(J(DRAFT))]))
text = llm1.chat([{"role": "user", "content": single_prompt}], max_tokens=800).text
single = parse_itinerary(text)
RESULTS["single-shot"] = dict(score=score(single), llm=llm1.n, tools=0, viol=validate(single))
print(*RESULTS["single-shot"]["viol"], sep="\n"); print("score:", RESULTS["single-shot"]["score"])
assert not is_live() and RESULTS["single-shot"]["score"] < 1 or is_live()

Notice the *kind* of errors: not ignorance but **bookkeeping**: adding up costs, driving time and day-specific rules across 9 activities. These are exactly the errors that tools and verifiers are good at, and that more prose is not.

### 3.2 · ReAct: think, act, observe (the AG02 loop with a reasoning habit)
The agent starts knowing only the brief. It decides to fetch the forecast (three parallel calls), then the catalog, then a travel time, reasoning in the text block before each call (that text is the "Thought"; the `tool_use` block is the "Action"; the `tool_result` is the "Observation").

In [ ]:
REACT_ANSWER = {1: ["shotover_jet", "skyline_luge", "burger_lunch"], 2: ["lakes_museum", "pottery_workshop"], 3: ["kawarau_bungy", "wine_tour"]}
llm2 = Counted(get_llm(script=[
    calls(("get_weather", {"day": 1}), ("get_weather", {"day": 2}), ("get_weather", {"day": 3}),
          _say="Thought: outdoor choices depend on the forecast, so I check all three days first."),
    call("list_activities", _say="Thought: now I need the options, with hours, costs and opening days."),
    call("travel_minutes", from_zone="town", to_zone="arrowtown", _say="Thought: Arrowtown days need driving time; checking the shortest leg."),
    say(J(REACT_ANSWER)),
]))
SYS_REACT = "You plan trips. Think briefly before each tool call. Use tools to get facts rather than recalling them. Finish with the JSON only."
res = run_agent(llm2, DATA_TOOLS, BRIEF, system=SYS_REACT, max_turns=8)
res.tracer.show()
react = parse_itinerary(res.final_text)
RESULTS["ReAct"] = dict(score=score(react), llm=llm2.n, tools=res.tracer.n_tool_calls(), viol=validate(react))
print(*RESULTS["ReAct"]["viol"], sep="\n"); print("score:", RESULTS["ReAct"]["score"])
assert res.stop == "end_turn" and res.tracer.n_tool_calls("get_weather") == 3

ReAct used facts instead of recall: it saw the rain and avoided outdoor activities on day 2. But no step ever *added up the budget*, and nothing checked the final answer, so a cost violation slipped through. ReAct's weakness is the absence of a global check.

### 3.3 · Plan-and-execute
The **planner** writes the full step list as data. Because a plan is data, we can **validate it before spending anything**: unique ids, dependencies that point backwards, only known tools. Then a dumb **executor** runs it, binding `"$s5"`-style references to earlier results, and the last step is a verifier.

In [ ]:
def validate_plan(plan: list, tools: ToolBox) -> list:
    errs, seen = [], set()
    for s in plan:
        if s["id"] in seen: errs.append(f"duplicate step id {s['id']}")
        for d in s.get("deps", []):
            if d not in seen: errs.append(f"step {s['id']} depends on {d}, which does not come earlier")
        if s["tool"] not in ("LLM", *tools.tools): errs.append(f"step {s['id']} uses unknown tool '{s['tool']}'")
        for v in s.get("args", {}).values():
            if isinstance(v, str) and v.startswith("$") and v[1:] not in seen: errs.append(f"step {s['id']} references {v} before it exists")
        seen.add(s["id"])
    return errs

PLAN = [
    {"id": "s1", "goal": "forecast day 1", "tool": "get_weather", "args": {"day": 1}, "deps": []},
    {"id": "s2", "goal": "forecast day 2", "tool": "get_weather", "args": {"day": 2}, "deps": []},
    {"id": "s3", "goal": "forecast day 3", "tool": "get_weather", "args": {"day": 3}, "deps": []},
    {"id": "s4", "goal": "fetch the activity catalog", "tool": "list_activities", "args": {}, "deps": []},
    {"id": "s5", "goal": "assemble an itinerary satisfying the brief from the facts", "tool": "LLM", "args": {}, "deps": ["s1", "s2", "s3", "s4"]},
    {"id": "s6", "goal": "verify the itinerary against the brief", "tool": "check_itinerary", "args": {"itinerary_json": "$s5"}, "deps": ["s5"]},
]
assert validate_plan(PLAN, ALL_TOOLS) == []
# the validator catches the planner mistakes we care about
assert validate_plan([{"id": "a", "goal": "", "tool": "book_hotel", "args": {}, "deps": []}], ALL_TOOLS)
assert validate_plan([{"id": "a", "goal": "", "tool": "get_weather", "args": {}, "deps": ["b"]}], ALL_TOOLS)
assert validate_plan([{"id": "a", "goal": "", "tool": "check_itinerary", "args": {"itinerary_json": "$z"}, "deps": []}], ALL_TOOLS)

def execute_plan(llm, plan, toolbox, task, system=""):
    state, n_tool = {}, 0
    for s in plan:
        args = {k: state[v[1:]] if isinstance(v, str) and v.startswith("$") else v for k, v in s["args"].items()}
        if s["tool"] == "LLM":
            facts = "\n".join(f"{d}: {state[d][:600]}" for d in s["deps"])
            state[s["id"]] = llm.chat([{"role": "user", "content": f"{task}\n\nFacts gathered:\n{facts}\n\nTask: {s['goal']}. JSON only."}], system=system, max_tokens=800).text
        else:
            out, err = toolbox.run(s["tool"], args); n_tool += 1
            if err: return state, n_tool, f"step {s['id']} failed: {out}"      # a real system would re-plan here
            state[s["id"]] = out
    return state, n_tool, None

PLAN_ANSWER = {1: ["shotover_jet", "skyline_luge", "burger_lunch"], 2: ["lakes_museum", "pottery_workshop"], 3: ["lakes_museum", "kawarau_bungy", "wine_tour"]}
llm3 = Counted(get_llm(script=[say(json.dumps(PLAN)), say(J(PLAN_ANSWER))]))
plan_text = llm3.chat([{"role": "user", "content": f"{BRIEF}\n\nWrite a plan as JSON list of steps with id, goal, tool (one of {list(ALL_TOOLS.tools)} or LLM), args, deps. Use \"$<id>\" to pass an earlier result."}],
                      system="You are a planner. Output only JSON.", max_tokens=1000).text
plan = json.loads(re.search(r"\[.*\]", plan_text, re.S).group(0))
problems = validate_plan(plan, ALL_TOOLS); print("plan problems:", problems or "none")
assert problems == []
state, n_tool, failure = execute_plan(llm3, plan, ALL_TOOLS, BRIEF)
pe = parse_itinerary(state["s5"])
print("verifier output at the end of the plan:\n" + state["s6"])
RESULTS["plan-execute"] = dict(score=score(pe), llm=llm3.n, tools=n_tool, viol=validate(pe))
assert failure is None and n_tool == 5 and llm3.n == 2
assert state["s6"] == "\n".join(str(x) for x in validate(pe)) or state["s6"] == "OK"   # the in-plan verifier agrees with our scorer

Plan-and-execute made the cost *predictable* (always 2 model calls + 5 tool calls) and put a verifier at the end, so the failure is **detected** ("closed on day 3"). But it still *ships* a flawed itinerary: there is no loop back. Detection without a repair path is a smoke alarm in an empty house: useful for logging and for escalating to a human, not for the user.

### 3.4 · Reflection grounded in the verifier
The loop: draft → **verify (tool)** → if violations, hand the model the *located* violation text and ask for a revision → repeat, **bounded** by a round limit and a cycle check (an itinerary seen before means we are oscillating).

OFFLINE the reviser is a small local-search program that acts only when the message contains verifier output (`[C#]` lines): it greedily applies the single-activity swap/add/remove that most reduces violations, at most two moves per round, mimicking "fix the most salient issues first". LIVE it is Claude reading the same text.

In [ ]:
def patch(itin, max_moves=2):
    """Offline stand-in for a reviser: greedy local search on (violations, cost)."""
    itin = {d: list(v) for d, v in itin.items()}
    def key(i): return (len(validate(i)), sum(ACTS[a]["cost_pp"] for d in DAYS for a in i[d] if a in ACTS))
    for _ in range(max_moves):
        if not validate(itin): break
        best, best_k = None, key(itin)
        for d in DAYS:
            cand = [{**itin, d: [x for j, x in enumerate(itin[d]) if j != i]} for i in range(len(itin[d]))]          # remove one
            cand += [{**itin, d: itin[d] + [a]} for a in ACTS if a not in itin[d]]                                     # add one
            cand += [{**itin, d: itin[d][:i] + [a] + itin[d][i + 1:]} for i in range(len(itin[d])) for a in ACTS if a not in itin[d]]   # swap one
            for c in cand:
                if key(c) < best_k: best, best_k = c, key(c)
        if best is None: break
        itin = best
    return itin

def reviser_policy(messages, system, tools):
    last = messages[-1]["content"]
    cur = parse_itinerary(re.search(r"Current itinerary: (\{.*\})\n", last).group(1))
    if not re.search(r"\[C\d\]", last):                      # no external signal -> a model tends to say "looks fine"
        return say(J(cur))
    return say(J(patch(cur)))

def reflect(llm, draft, max_rounds=4, grounded=True):
    """Returns (final_itinerary, history, stop_reason, n_verifier_calls)."""
    cur, hist, seen, n_ver = draft, [], {J(draft)}, 0
    for r in range(1, max_rounds + 1):
        if grounded:
            report, _ = ALL_TOOLS.run("check_itinerary", {"itinerary_json": J(cur)}); n_ver += 1
        else:
            report = "Review this itinerary critically and improve it if needed."
        hist.append((r, report))
        if report == "OK": return cur, hist, "verified", n_ver
        nxt = parse_itinerary(llm.chat([{"role": "user", "content": f"{BRIEF}\n\nCurrent itinerary: {J(cur)}\n\nFeedback:\n{report}\n\nRevise to fix the feedback. JSON only."}], max_tokens=800).text)
        if J(nxt) in seen: return nxt, hist, "cycle", n_ver
        seen.add(J(nxt)); cur = nxt
    return cur, hist, "max_rounds", n_ver

llm4 = Counted(get_llm(policy=reviser_policy))
final, hist, why, n_ver = reflect(llm4, single)               # start from the single-shot draft (its 1 call is counted below)
for r, rep in hist: print(f"--- round {r} verifier:\n{rep}")
print("stopped:", why, "| final:", final)
RESULTS["reflect+verifier"] = dict(score=score(final), llm=1 + llm4.n, tools=n_ver, viol=validate(final))
assert why == "verified" and validate(final) == [] and 1 <= llm4.n <= 4

## 4 · Use it: score the strategies side by side
One number is not enough: a strategy that scores 1.0 with 12 model calls may lose to one that scores 0.9 with 2 (AG14 formalises this as cost-per-success). Tabulate quality *and* cost.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print(f"{'strategy':<18}{'score':>6}{'LLM calls':>11}{'tool calls':>12}   violations")
for k, r in RESULTS.items():
    print(f"{k:<18}{r['score']:>6.2f}{r['llm']:>11}{r['tools']:>12}   {sorted({x.code for x in r['viol']}) or '-'}")
fig, ax = plt.subplots(figsize=(5.6, 3.4))
for k, r in RESULTS.items():
    ax.scatter(r["llm"], r["score"], s=90, zorder=3); ax.annotate(k, (r["llm"], r["score"]), textcoords="offset points", xytext=(6, -10), fontsize=8)
ax.set_xlabel("model calls"); ax.set_ylabel("constraints satisfied (of 8)"); ax.set_ylim(0.5, 1.05); ax.set_xlim(0, 6.5); ax.grid(alpha=.3)
ax.set_title(("LIVE" if is_live() else "OFFLINE (scripted)") + ": quality vs cost"); plt.show()
if not is_live():       # scripted traces: these pin the harness, not any model
    assert RESULTS["single-shot"]["score"] < RESULTS["ReAct"]["score"] <= RESULTS["plan-execute"]["score"] < RESULTS["reflect+verifier"]["score"] == 1.0
    assert RESULTS["plan-execute"]["llm"] == 2 and RESULTS["reflect+verifier"]["llm"] > RESULTS["single-shot"]["llm"]
    assert RESULTS["reflect+verifier"]["llm"] <= 4        # the loop converged inside its budget

**Reading the table (what transfers to live runs).** Quality came from the **verifier**, not the pattern's name: ReAct and plan-and-execute improved on single-shot by *fetching facts*; only the strategy that **fed located verifier output back** reached 1.0. Cost is the flip side: reflection spent the most model calls, and its count depends on the input (it is an agent-style loop), whereas plan-and-execute was fixed at 2.

## 5 · Break it
### 5a · Reflection without evidence
Same loop, but the critique prompt has no verifier output ("review critically and improve"). The offline reviser mirrors what the literature reports: with no external signal, models are often unable to *find* their own errors, and may even degrade a correct answer (Huang et al., 2023). It is a stand-in, not a proof; run LIVE to see your model's behaviour.

In [ ]:
llm5 = Counted(get_llm(policy=reviser_policy))
ungrounded, hist5, why5, _ = reflect(llm5, single, max_rounds=3, grounded=False)
print("ungrounded reflection stopped:", why5, "| score:", score(ungrounded), "(started at", score(single), ")", "| model calls:", llm5.n)
if not is_live():
    assert score(ungrounded) == score(single) and why5 in ("cycle", "max_rounds")      # spent calls, gained nothing
    assert llm5.n >= 1

### 5b · Oscillation: fix one thing, break another
Revisions can flip-flop between two flawed answers. The `seen` set in `reflect` catches a repeated itinerary and stops, instead of burning the budget. Here a pathological reviser alternates between two drafts:

In [ ]:
A = {1: ["shotover_jet", "skyline_luge"], 2: ["kawarau_bungy", "lakes_museum"], 3: ["routeburn_hike", "spa_afternoon"]}
B = {1: ["shotover_jet", "skyline_luge"], 2: ["pottery_workshop", "lakes_museum"], 3: ["routeburn_hike", "kiwi_birdlife_park", "spa_afternoon"]}
flip = Counted(get_llm(script=[say(J(B)), say(J(A)), say(J(B)), say(J(A))]))
out, h, why, _ = reflect(flip, A, max_rounds=6)
print("stopped:", why, "after", flip.n, "revisions (budget was 6)")
assert why == "cycle" and flip.n <= 3 and validate(out) != []

### 5c · The plan goes stale
Plan-and-execute commits to facts gathered early. If the world changes during execution (the day-3 forecast flips to rain after step `s3`), the executor happily carries on. Only the *final verifier* notices, and a re-plan is needed:

In [ ]:
WEATHER[3] = "rain"                       # simulated forecast update after planning
stale = validate(pe)
print(*stale, sep="\n")
assert any(x.code == "C3" and x.day == 3 for x in stale)    # outdoor bungy on day 3 is now invalid
fixed = patch(pe, max_moves=6)
WEATHER[3] = "clear"
print("after re-planning against the new forecast:", fixed)
WEATHER[3] = "rain"; assert validate(fixed) == []; WEATHER[3] = "clear"
assert validate(pe) != []                 # but the original plan is still flawed under the original forecast (C7)

*Design rules that fall out of 5a-5c:* (1) every loop has a **round limit and a cycle check**; (2) critique must be **grounded** in something the model cannot argue with; (3) put a **verification step at the end of any plan** and a **re-plan path** when it fails; (4) re-fetch volatile facts close to where they are used.

## 6 · When do thinking models replace the scaffold?
Modern Claude models can reason before answering (*extended thinking*), and for tool-using agents they can reason between tool calls. That changes the economics:

| You hand-built… | A thinking model… | Keep the scaffold when… |
|---|---|---|
| "Think step by step" prompting / chain-of-thought | does it internally, usually better | you need the reasoning in a specific *format* for audit |
| A planner step for short tasks | often plans implicitly in its thinking | the plan must be **approved by a human** or **executed by cheaper workers / in parallel** |
| Reflection with *self*-critique | still bounded by what it can notice | you have an **external verifier**: that signal is worth more than extra thinking |
| A step budget | can over-think easy tasks | you must **cap cost and latency** deterministically |

Reference sketch, *not run here; check current docs for parameter names and supported models*:
```python
resp = client.messages.create(model=MODEL, max_tokens=16000,
                              thinking={"type": "enabled", "budget_tokens": 8000},   # budget = ceiling on reasoning tokens
                              tools=tools, messages=messages)
# In tool loops, return the assistant's thinking blocks unchanged along with its tool_use blocks (see AG02).
```
**The decision rule is empirical:** run your eval with the scaffold and without, pay attention to *cost per success* as well as success rate, and keep a scaffold only if it moves a metric by more than run-to-run noise (AG14 builds that harness). Intuition about "smarter models don't need scaffolds" ages quickly, in both directions.

## 7 · Exercises
**Practice (30 min)**
1. Add a ninth constraint **C9**: *at most one activity per day may cost more than $100.* Implement it in `validate`, re-check that `GOLD` still exists (re-run `solve()`), and see which strategies' scores change.
2. Make `validate_plan` also reject a plan with **more than 8 steps** or with any step whose `goal` is empty. Add two failing-plan assertions.

**Stretch (60 min)**
3. Give plan-and-execute a **repair path**: if the final verifier reports violations, call `reflect()` starting from the plan's itinerary. Compare score and total model calls with plain `reflect` from the single-shot draft. When is each cheaper?
4. Add **weather volatility**: in `execute_plan`, re-check the forecast just before the synthesis step and abort with a re-plan request if it changed. Assert that §5c's stale plan can no longer be returned silently.

**Challenge (2 h, a key helps)**
5. Create 10 random *brief variants* (different budgets 400-700, rain on different days, different closed days). Run single-shot, ReAct, and reflect-with-verifier LIVE on all 10 (fix `AGENTKIT_MODEL` for a small and a large model). Report mean score, mean model calls and cost per fully-correct itinerary (use `agentkit.trace.PRICES` and the tracer's token counts). Does the larger model with *no* scaffold beat the smaller model *with* reflection? State your evidence and its limits (n = 10).

In [ ]:
# Starter for exercise 1: extend validate() (copy the function, add the rule) and re-run solve()
def violations_c9(itin) -> list:
    """C9: at most one activity per day may cost more than $100."""
    raise NotImplementedError

# for d in DAYS: assert len([a for a in GOLD[d] if ACTS[a]["cost_pp"] > 100]) <= 1 or violations_c9(GOLD)

## ✅ Checkpoint (answer before moving on)
1. Why did the ReAct run miss the budget violation even though it used tools correctly?
2. Plan-and-execute detected a violation but still returned a wrong itinerary. What two things would you add so it either fixes it or fails safely?
3. Why does reflection without a verifier often fail to improve answers, and what are three kinds of verifier you could substitute for a trip planner?
4. A new thinking model scores 96 % on your eval with no scaffold; your plan-and-reflect pipeline scores 97 % at 4× the cost. What do you do?

<details><summary>Answers</summary>

1. No step computed or checked the total; ReAct only gathers facts and reasons in prose, and arithmetic over many items is where unaided models slip. A tool (`check_itinerary`) or a calculator step would catch it.
2. A **repair path** (feed the located violations into a bounded revise loop, or re-plan) and a **safe failure** (if still violated after N rounds, return the flaw explicitly or escalate to a human instead of presenting the itinerary as valid).
3. The same model that made the error judges its own text, with no new evidence, so it tends to reaffirm the answer. Substitutes: executable constraint checker (ours), a **booking-system availability API** (real opening days and prices), a **human planner's approval**, or a separately-prompted judge checked against labelled examples.
4. Compare *cost per success* and the confidence interval: a 1-point gain on a small eval can be noise. Likely keep the simple pipeline, and keep only the *verifier* (cheap code) as a safety net. Re-run when models or briefs change.
</details>

## 📌 Key takeaways
- **ReAct** lets observations steer decisions; **plan-and-execute** makes steps auditable and predictable; **reflection** repeats until a bar is met. All of them gain most from **external verification**.
- Encode the brief as a **verifier** (located violations); it doubles as scorer, tool and critique signal.
- Validate plans as data before executing; give them dependencies, a final check, and a re-plan path.
- Bound every loop (rounds, cycle detection, budgets) and **measure quality and cost together**.
- A thinking model replaces *prompting tricks* more than it replaces *verification, approvals and budgets*. Decide by ablation.

## 📚 Further reading
- Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models* (2022), arXiv:2210.03629
- Wei et al., *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* (2022), arXiv:2201.11903
- Shinn et al., *Reflexion: Language Agents with Verbal Reinforcement Learning* (2023), arXiv:2303.11366
- Madaan et al., *Self-Refine: Iterative Refinement with Self-Feedback* (2023), arXiv:2303.17651
- Huang et al., *Large Language Models Cannot Self-Correct Reasoning Yet* (2023), arXiv:2310.01798
- Yao et al., *Tree of Thoughts: Deliberate Problem Solving with Large Language Models* (2023), arXiv:2305.10601
- Anthropic Engineering, *Building effective agents* — anthropic.com/engineering/building-effective-agents

**Next → AG05 · Memory & Context Engineering** (keep a long-running agent inside its context budget).